# Alignment and cameras

Kabsch aligns two clouds when you already know which point matches which point. ICP is what you do when you do not. Cameras matter here because the product ingests imagery with the scans, but the geometry above is the main event.

Camera frame used below: +Z looks forward, `u` is the column, `v` is the row, and `image_shape` is `(height, width)`.


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


## 1. Kabsch

`P` and `Q` are corresponding points, same length, same order. Find `R` and `t` with `det(R) = +1` such that `Q ≈ (R @ P.T).T + t`.

This function does not search for matches. If the best linear map is a reflection, flip it back to a rotation. A mirrored point set will then not fit perfectly, and that is the correct behavior: a rigid motion cannot mirror a cloud.

Hand check: build a known `R` and `t`, set `Q = (R @ P.T).T + t`, and recover them. Also run a cloud reflected through `z → -z` and assert the returned determinant is positive.

<details><summary>Hint</summary>

Center both clouds. Covariance `H = Pc.T @ Qc` is 3×3. `U, _, Vt = np.linalg.svd(H)`, then `R = Vt.T @ U.T`. If `det(R) < 0`, flip the last row of `Vt` (the singular vector for the smallest singular value) and recompute `R`. Translation is `t = mean(Q) - R @ mean(P)`.

</details>


In [ ]:
import numpy as np

def kabsch(P, Q):
    """
    Rigid alignment of corresponding points. p maps toward q.

    P and Q are (N, 3), same order, same count. Correspondences are given.
    Returns R, t with Q ≈ (R @ P.T).T + t and det(R) = +1.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def kabsch(P, Q):
    """Rigid alignment of corresponding points. p maps toward q.

    P and Q are (N, 3), same order, same count. Correspondences are given.
    Returns R, t with Q ≈ (R @ P.T).T + t and det(R) = +1.
    """
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    if len(P) != len(Q) or len(P) < 3:
        raise ValueError("P and Q must be corresponding sets of at least 3 points")
    cP = P.mean(axis=0)
    cQ = Q.mean(axis=0)
    H = (P - cP).T @ (Q - cQ)
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    # det < 0 is a reflection. Flip the smallest singular vector and rebuild.
    if np.linalg.det(R) < 0.0:
        Vt = Vt.copy()
        Vt[-1, :] *= -1.0
        R = Vt.T @ U.T
    t = cQ - R @ cP
    return R, t

def check_kabsch():
    rng = np.random.default_rng(0)
    P = rng.normal(size=(25, 3))
    c, s = np.cos(0.7), np.sin(0.7)
    Rz = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
    Rx = np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])
    R_true = Rz @ Rx
    t_true = np.array([1.0, -2.0, 0.5])
    Q = (R_true @ P.T).T + t_true
    R, t = kabsch(P, Q)
    assert np.allclose(R, R_true, atol=1e-8)
    assert np.allclose(t, t_true, atol=1e-8)
    assert np.linalg.det(R) > 0.0
    reflected = (np.diag([1.0, 1.0, -1.0]) @ P.T).T
    R_bad, _ = kabsch(P, reflected)
    assert np.linalg.det(R_bad) > 0.0
    print("passed")

check_kabsch()


## 2. One ICP iteration

One iteration, not a full solver. Match every source point to its nearest target point, then run Kabsch on those pairs. Return `R, t` that map source toward target.

Nearest neighbors are the true matches only when the motion is small compared with the spacing of the points. Say that assumption. A 90° rotation of a noisy scan will pair the wrong points, and one iteration cannot invent the right correspondence.

Hand check: four points at the origin and 10 units out on each axis, rotated by `0.05` rad about Z and translated by `(0.1, -0.2, 0.05)`. The points are far apart relative to the motion, so one iteration recovers `R` and `t`.

The reference cell includes nearest-neighbor and Kabsch helpers.


In [ ]:
import numpy as np

def icp_iteration(source, target):
    """
    One ICP iteration: match source→target by nearest neighbor, then Kabsch.

    Recovers a rigid motion only when the motion is small enough (or the points
    are separated enough) that nearest neighbors are the true matches.
    Returns R, t aligning source onto target.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

# Included so this cell runs by itself, including helpers from earlier problems.

def nearest_neighbors(A, B):
    """Brute-force nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    Memory is O(N*M). Chunk A, or use a KD-tree, when that product is large.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if len(B) == 0:
        raise ValueError("B is empty")
    delta = A[:, None, :] - B[None, :, :]
    dist = np.linalg.norm(delta, axis=-1)
    idx = np.argmin(dist, axis=1)
    return idx, dist[np.arange(len(A)), idx]

def kabsch(P, Q):
    """Rigid alignment of corresponding points. p maps toward q.

    P and Q are (N, 3), same order, same count. Correspondences are given.
    Returns R, t with Q ≈ (R @ P.T).T + t and det(R) = +1.
    """
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    if len(P) != len(Q) or len(P) < 3:
        raise ValueError("P and Q must be corresponding sets of at least 3 points")
    cP = P.mean(axis=0)
    cQ = Q.mean(axis=0)
    H = (P - cP).T @ (Q - cQ)
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    # det < 0 is a reflection. Flip the smallest singular vector and rebuild.
    if np.linalg.det(R) < 0.0:
        Vt = Vt.copy()
        Vt[-1, :] *= -1.0
        R = Vt.T @ U.T
    t = cQ - R @ cP
    return R, t

def icp_iteration(source, target):
    """One ICP iteration: match source→target by nearest neighbor, then Kabsch.

    Recovers a rigid motion only when the motion is small enough (or the points
    are separated enough) that nearest neighbors are the true matches.
    Returns R, t aligning source onto target.
    """
    idx, _ = nearest_neighbors(source, target)
    return kabsch(source, target[idx])

def check_icp_iteration():
    P = np.array(
        [
            [0.0, 0.0, 0.0],
            [10.0, 0.0, 0.0],
            [0.0, 10.0, 0.0],
            [0.0, 0.0, 10.0],
        ]
    )
    c, s = np.cos(0.05), np.sin(0.05)
    R_true = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
    t_true = np.array([0.1, -0.2, 0.05])
    Q = (R_true @ P.T).T + t_true
    R, t = icp_iteration(P, Q)
    assert np.allclose(R, R_true, atol=1e-8)
    assert np.allclose(t, t_true, atol=1e-8)
    print("passed")

check_icp_iteration()


## 3. Project points into an image

Project world points with extrinsics `(R, t)` and a 3×3 intrinsic matrix `K`.

Camera coordinates are `p_c = R @ p + t`, and the camera looks down **+Z** (a computer-vision camera, not the OpenGL camera that looks down −Z). Pixels are `u, v = (K @ p_c)[:2] / z`.

Return all pixel coordinates and a mask of points that are in front of the camera (`z > 0`) and inside the image. `image_shape` is `(height, width)`. Use half-open bounds: `0 <= u < width`, `0 <= v < height`. Set pixels behind the camera to NaN rather than dividing by a non-positive z.

Hand check: `K = [[200, 0, 160], [0, 200, 120], [0, 0, 1]]`, `R = I`, `t = 0`, image `(240, 320)`.

- `(0, 0, 10)` → `(160, 120)`, valid
- `(1, 0, 10)` → `(180, 120)`, valid
- `(0, 0, -5)` invalid
- `(10, 0, 10)` is in front but `u = 360` is outside


In [ ]:
import numpy as np

def project_points(points, K, R, t, image_shape):
    """
    Project world points with extrinsics (R, t) and intrinsics K.

    Camera coordinates are p_c = R @ p_w + t, looking down +Z.
    A point is kept when z > 0 and the pixel lies inside the image.
    image_shape is (height, width). Pixel u is the column, v is the row.
    returns uv (N, 2), valid (N,). uv is meaningless where valid is False
    (including points at or behind the camera, where z is not positive).
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def project_points(points, K, R, t, image_shape):
    """Project world points with extrinsics (R, t) and intrinsics K.

    Camera coordinates are p_c = R @ p_w + t, looking down +Z.
    A point is kept when z > 0 and the pixel lies inside the image.
    image_shape is (height, width). Pixel u is the column, v is the row.
    returns uv (N, 2), valid (N,). uv is meaningless where valid is False
    (including points at or behind the camera, where z is not positive).
    """
    points = np.asarray(points, dtype=float)
    K = np.asarray(K, dtype=float)
    t = np.asarray(t, dtype=float)
    height, width = image_shape
    camera = (R @ points.T).T + t
    z = camera[:, 2]
    projected = (K @ camera.T).T
    uv = np.full((len(points), 2), np.nan)
    in_front = z > 0.0
    uv[in_front] = projected[in_front, :2] / z[in_front, None]
    valid = (
        in_front
        & (uv[:, 0] >= 0.0)
        & (uv[:, 0] < width)
        & (uv[:, 1] >= 0.0)
        & (uv[:, 1] < height)
    )
    return uv, valid

def check_project_points():
    K = np.array([[200.0, 0.0, 160.0], [0.0, 200.0, 120.0], [0.0, 0.0, 1.0]])
    pts = np.array(
        [
            [0.0, 0.0, 10.0],
            [1.0, 0.0, 10.0],
            [0.0, 0.0, -5.0],
            [10.0, 0.0, 10.0],
        ]
    )
    uv, valid = project_points(pts, K, np.eye(3), np.zeros(3), (240, 320))
    assert np.array_equal(valid, [True, True, False, False])
    assert np.allclose(uv[0], [160, 120])
    assert np.allclose(uv[1], [180, 120])
    print("passed")

check_project_points()


## 4. Back-project a depth map

`depth` is `(H, W)`, the z-coordinate in the camera frame, not a ray length along a non-unit direction. `K` is 3×3. Return an `(N, 3)` cloud of the pixels whose depth is positive. Order is row-major: `depth[v, u]` is row `v`, column `u`.

Hand check: the same `K` as the projection problem, `depth[120, 180] = 10`, and every other pixel 0. The only returned point is `(1, 0, 10)`.

<details><summary>Hint</summary>

Stack pixel coordinates `(u, v, 1)`, multiply by `K⁻¹` on the right if the pixels are rows (`pixels @ inv(K).T`), and scale by depth. Inverting `K` once is fine. Prefer `solve` when the same inverse would be applied inside a loop to independent right-hand sides; here there is one 3×3 matrix.

</details>


In [ ]:
import numpy as np

def depth_to_points(depth, K):
    """
    Back-project a depth map to a camera-frame point cloud.

    depth: (H, W), meters along the camera +Z axis. K: (3, 3).
    Pixels with depth <= 0 are dropped. Output is (N, 3), row-major.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def depth_to_points(depth, K):
    """Back-project a depth map to a camera-frame point cloud.

    depth: (H, W), meters along the camera +Z axis. K: (3, 3).
    Pixels with depth <= 0 are dropped. Output is (N, 3), row-major.
    """
    depth = np.asarray(depth, dtype=float)
    K = np.asarray(K, dtype=float)
    vs, us = np.indices(depth.shape)
    pixels = np.stack([us, vs, np.ones_like(us)], axis=-1).astype(float)
    rays = pixels @ np.linalg.inv(K).T
    points = rays * depth[..., None]
    return points[depth > 0.0]

def check_depth_to_points():
    K = np.array([[200.0, 0.0, 160.0], [0.0, 200.0, 120.0], [0.0, 0.0, 1.0]])
    depth = np.zeros((240, 320))
    depth[120, 180] = 10.0
    depth[40, 160] = 0.0
    pts = depth_to_points(depth, K)
    assert pts.shape == (1, 3)
    assert np.allclose(pts[0], [1.0, 0.0, 10.0])
    print("passed")

check_depth_to_points()
